In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pasta_ipca = BASE_DIR / "data" / "bronze" / "ipca"

arquivos = list(pasta_ipca.glob("*.json"))
assert len(arquivos) > 0, "Nenhum arquivo encontrado em data/bronze/ipca/"

In [2]:
df_raw = pd.read_json(arquivos[0])
print(f"Arquivo: {arquivos[0].name} | Linhas: {len(df_raw)}")
df_raw.head()

Arquivo: ipca_2026-09-06.json | Linhas: 59


,data,valor
0,01/09/2021,1.16
1,01/10/2021,1.25
2,01/11/2021,0.95
3,01/12/2021,0.73
4,01/01/2022,0.54


In [3]:
df = df_raw.copy()
df = df.rename(columns={"valor": "variacao_ipca_mensal"})

In [4]:
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y").dt.to_period("M").dt.to_timestamp()
df["variacao_ipca_mensal"] = pd.to_numeric(df["variacao_ipca_mensal"], errors="coerce")
df = df.sort_values("data").reset_index(drop=True)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   data                  59 non-null     datetime64[us]
 1   variacao_ipca_mensal  59 non-null     float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 1.1 KB


In [5]:
assert not df["data"].duplicated().any(), "Erro: Meses duplicados detectados no IPCA!"
assert df["variacao_ipca_mensal"].isnull().sum() == 0, "Erro: Valores nulos encontrados no IPCA!"

In [6]:
# Valida se os intervalos são estritamente de 1 mês
diffs_dias = df["data"].diff().dropna().dt.days
print(f"Intervalo mínimo entre meses: {diffs_dias.min()} dias | Intervalo máximo: {diffs_dias.max()} dias")

Intervalo mínimo entre meses: 28 dias | Intervalo máximo: 31 dias


In [7]:
df["ipca_acumulado_12m"] = df["variacao_ipca_mensal"].rolling(12).sum().round(2)
display(df.tail(12))

,data,variacao_ipca_mensal,ipca_acumulado_12m
47,2025-08-01,-0.11,5.02
48,2025-09-01,0.48,5.06
49,2025-10-01,0.09,4.59
50,2025-11-01,0.18,4.38
51,2025-12-01,0.33,4.19
52,2026-01-01,0.33,4.36
53,2026-02-01,0.70,3.75
54,2026-03-01,0.88,4.07
55,2026-04-01,0.67,4.31
56,2026-05-01,0.58,4.63


In [8]:
# Diretório de destino na camada Silver
pasta_silver = BASE_DIR / "data" / "silver"
pasta_silver.mkdir(parents=True, exist_ok=True)

# Gravação do Parquet de teste
caminho_teste_parquet = pasta_silver / "ipca_silver_test.parquet"
df.to_parquet(caminho_teste_parquet, index=False)

print("Teste de escrita Parquet concluído com sucesso!")
print(f"Arquivo salvo em: {caminho_teste_parquet.relative_to(BASE_DIR)}")

Teste de escrita Parquet concluído com sucesso!
Arquivo salvo em: data/silver/ipca_silver_test.parquet
